# QM 640 Capstone — Step 2: Index Constituent Lists (Sector + Firm Size)

Pulls S&P 500 constituents with GICS sector (Wikipedia), Russell 3000 via
the iShares IWV public holdings CSV, and market capitalization per ticker
via yfinance. Feeds RQ3 (firm size) and RQ4 (sector).

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [1]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 895, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 895 (delta 35), reused 49 (delta 19), pack-reused 813 (from 1)
Receiving objects: 100% (895/895), 5.44 MiB | 5.64 MiB/s, done.
Resolving deltas: 100% (476/476), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [2]:
!pip install -q pandas numpy requests yfinance lxml html5lib

## Cell 3 — Configuration

In [3]:
import os

OUTPUT_DIR = os.path.join(BASE_DIR, "data/raw")
HEADERS = {"User-Agent": "Mozilla/5.0 (QM640 Capstone research; Shan_muganathan@yahoo.com)"}

print("Will write output to:", OUTPUT_DIR)

Will write output to: /content/QM640-WALSH-CAPSTONE/data/raw


## Cell 4 — S&P 500 constituents (GICS sector)

In [4]:
import pandas as pd


def get_sp500_constituents():
    """Pull S&P 500 list + GICS sector from Wikipedia's maintained table."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    tables = pd.read_html(url, storage_options=HEADERS)
    df = tables[0]
    df = df.rename(columns={
        "Symbol": "ticker",
        "Security": "company_name",
        "GICS Sector": "gics_sector",
        "GICS Sub-Industry": "gics_sub_industry",
    })
    df["ticker"] = df["ticker"].str.replace(".", "-", regex=False)  # BRK.B -> BRK-B for yfinance
    keep = ["ticker", "company_name", "gics_sector", "gics_sub_industry"]
    df = df[[c for c in keep if c in df.columns]]
    out_path = os.path.join(OUTPUT_DIR, "sp500_constituents.csv")
    df.to_csv(out_path, index=False)
    print(f"S&P 500: {len(df)} constituents -> {out_path}")
    return df


sp500 = get_sp500_constituents()
sp500.head()

S&P 500: 503 constituents -> /content/QM640-WALSH-CAPSTONE/data/raw/sp500_constituents.csv


,ticker,company_name,gics_sector,gics_sub_industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services


## Cell 5 — Russell 3000 constituents (via iShares IWV holdings)

In [5]:
import requests
from io import StringIO


def get_russell3000_constituents():
    """
    Pull Russell 3000 constituents via iShares Russell 3000 ETF (IWV) public
    holdings CSV. iShares publishes this at a stable URL, updated daily.
    """
    url = (
        "https://www.ishares.com/us/products/239714/ishares-russell-3000-etf/"
        "1467271812596.ajax?fileType=csv&fileName=IWV_holdings&dataType=fund"
    )
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    lines = resp.text.splitlines()
    header_idx = next(i for i, l in enumerate(lines) if l.startswith("Ticker,"))
    df = pd.read_csv(StringIO("\n".join(lines[header_idx:])), thousands=",")

    df = df.rename(columns={
        "Ticker": "ticker",
        "Name": "company_name",
        "Sector": "sector",
        "Weight (%)": "weight_pct",
    })
    keep = ["ticker", "company_name", "sector", "weight_pct"]
    df = df[[c for c in keep if c in df.columns]]
    df = df.dropna(subset=["ticker"])
    out_path = os.path.join(OUTPUT_DIR, "russell3000_constituents.csv")
    df.to_csv(out_path, index=False)
    print(f"Russell 3000 (via IWV): {len(df)} constituents -> {out_path}")
    return df


try:
    russell = get_russell3000_constituents()
except Exception as e:
    print(f"Russell 3000 pull failed ({e}) - S&P 500 alone is an acceptable "
          f"sampling frame if this URL has changed; check https://www.ishares.com "
          f"for the current IWV holdings link.")
    russell = pd.DataFrame(columns=["ticker"])

russell.head()

Russell 3000 pull failed () - S&P 500 alone is an acceptable sampling frame if this URL has changed; check https://www.ishares.com for the current IWV holdings link.


,ticker


## Cell 5b — Unified sector reference (GICS priority, Russell fallback)

**Fix (Interim Report feedback — Dataset Completeness / Limited Dataset Diversity):** `05_market_model_car.ipynb` previously matched sector using only `sp500_constituents.csv`'s GICS sector, silently dropping every Russell-3000-only firm even though `russell["sector"]` was already being collected right here and never used. This merges both into one reference file, GICS first (finer, standardized taxonomy), Russell's sector as a fallback only where no GICS match exists.

In [6]:
sp500_sector = sp500[["ticker", "gics_sector"]].rename(columns={"gics_sector": "sector"}).copy()
sp500_sector["sector_source"] = "GICS (S&P 500)"

if "sector" in russell.columns and not russell.empty:
    russell_sector = russell[["ticker", "sector"]].dropna(subset=["sector"]).copy()
    russell_sector["sector_source"] = "Russell 3000 (iShares)"
else:
    russell_sector = pd.DataFrame(columns=["ticker", "sector", "sector_source"])

# GICS rows first -> on ticker overlap, drop_duplicates(keep="first") lets GICS win
combined = pd.concat([sp500_sector, russell_sector], ignore_index=True)
combined = combined.dropna(subset=["sector"])
sector_reference = combined.drop_duplicates(subset=["ticker"], keep="first")

out_path = os.path.join(OUTPUT_DIR, "sector_reference.csv")
sector_reference.to_csv(out_path, index=False)

n_gics = (sector_reference["sector_source"] == "GICS (S&P 500)").sum()
n_russell = (sector_reference["sector_source"] == "Russell 3000 (iShares)").sum()
print(f"Unified sector reference: {len(sector_reference)} tickers "
      f"({n_gics} GICS, {n_russell} Russell-only) -> {out_path}")
print("NOTE: this is still an index-membership reference (fast, broad-coverage "
      "first pass). Confirmed event tickers outside BOTH S&P 500 and Russell "
      "3000 will still be missing here - Step 5 (05_market_model_car.ipynb) "
      "now gap-fills those directly per ticker, so this file no longer needs "
      "to have 100% coverage of your event tickers to be useful.")
sector_reference.head()

Unified sector reference: 503 tickers (503 GICS, 0 Russell-only) -> /content/QM640-WALSH-CAPSTONE/data/raw/sector_reference.csv
NOTE: this is still an index-membership reference (fast, broad-coverage first pass). Confirmed event tickers outside BOTH S&P 500 and Russell 3000 will still be missing here - Step 5 (05_market_model_car.ipynb) now gap-fills those directly per ticker, so this file no longer needs to have 100% coverage of your event tickers to be useful.


,ticker,sector,sector_source
0,MMM,Industrials,GICS (S&P 500)
1,AOS,Industrials,GICS (S&P 500)
2,ABT,Health Care,GICS (S&P 500)
3,ABBV,Health Care,GICS (S&P 500)
4,ACN,Information Technology,GICS (S&P 500)


## Cell 6 — Market capitalization per ticker (firm_size for RQ3)

In [7]:
import numpy as np
import yfinance as yf
import time


def get_market_cap(tickers, batch_size=50, pause=1.0):
    """Pull market cap per ticker via yfinance, log-transform for firm_size.
    Tries fast_info first (quick), falls back to the full .info dict if
    that fails or returns nothing - fast_info's shape has changed across
    yfinance versions before, so a single-method approach is fragile."""
    rows = []
    fast_failures = 0
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i + batch_size]
        for t in batch:
            cap = None
            try:
                fi = yf.Ticker(t).fast_info
                cap = fi.get("market_cap") if hasattr(fi, "get") else getattr(fi, "market_cap", None)
            except Exception:
                cap = None

            if not cap:
                fast_failures += 1
                try:
                    info = yf.Ticker(t).info
                    cap = info.get("marketCap")
                except Exception as e:
                    print(f"  skip {t}: {e}")

            if cap and cap > 0:
                rows.append({
                    "ticker": t,
                    "market_cap_usd": cap,
                    "market_cap_log": np.log(cap),
                })
        print(f"  processed {min(i + batch_size, len(tickers))}/{len(tickers)}")
        time.sleep(pause)

    if fast_failures > len(tickers) * 0.5:
        print(f"NOTE: fast_info failed for {fast_failures}/{len(tickers)} tickers - "
              f"fell back to .info (slower but more reliable) for those.")

    df = pd.DataFrame(rows)
    if df.empty:
        print("WARNING: no market cap data collected at all. This usually means "
              "yfinance itself is unreachable or rate-limited in this session, "
              "not a code bug - try again in a few minutes, or check "
              "`pip show yfinance` is a recent version.")

    out_path = os.path.join(OUTPUT_DIR, "firm_size.csv")
    df.to_csv(out_path, index=False)
    print(f"Market cap collected for {len(df)} tickers -> {out_path}")
    return df


all_tickers = sorted(set(sp500["ticker"]) | set(russell.get("ticker", pd.Series(dtype=str))))
print(f"Pulling market cap for {len(all_tickers)} unique tickers ...")
firm_size = get_market_cap(all_tickers)

Pulling market cap for 503 unique tickers ...
  processed 50/503
  processed 100/503
  processed 150/503
  processed 200/503
  processed 250/503
  processed 300/503
  processed 350/503
  processed 400/503
  processed 450/503
  processed 500/503
  processed 503/503
NOTE: fast_info failed for 503/503 tickers - fell back to .info (slower but more reliable) for those.
Market cap collected for 503 tickers -> /content/QM640-WALSH-CAPSTONE/data/raw/firm_size.csv


## Commit and push results back to GitHub

In [8]:
!git -C {BASE_DIR} add "data/raw/sp500_constituents.csv"
!git -C {BASE_DIR} add "data/raw/russell3000_constituents.csv"
!git -C {BASE_DIR} add "data/raw/sector_reference.csv"
!git -C {BASE_DIR} add "data/raw/firm_size.csv"
!git -C {BASE_DIR} commit -m "Step 2: index constituent lists (sector + firm size), add unified GICS+Russell sector reference"
!git -C {BASE_DIR} push


fatal: pathspec 'data/raw/russell3000_constituents.csv' did not match any files
[main 0895cdf] Step 2: index constituent lists (sector + firm size), add unified GICS+Russell sector reference
 3 files changed, 1512 insertions(+)
 create mode 100644 data/raw/firm_size.csv
 create mode 100644 data/raw/sector_reference.csv
 create mode 100644 data/raw/sp500_constituents.csv
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (7/7), 19.42 KiB | 19.42 MiB/s, done.
Total 7 (delta 0), reused 3 (delta 0), pack-reused 0
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   1efef25..0895cdf  main -> main


## Sanity check

In [9]:
print("S&P 500:", sp500.shape)
print("Russell 3000:", russell.shape)
print("Unified sector reference:", sector_reference.shape)
print("Firm size:", firm_size.shape)
firm_size.head()


S&P 500: (503, 4)
Russell 3000: (0, 1)
Unified sector reference: (503, 3)
Firm size: (503, 3)


,ticker,market_cap_usd,market_cap_log
0,A,39080103936,24.388879
1,AAPL,4537070911488,29.143303
2,ABBV,443359002624,26.817646
3,ABNB,89927868416,25.222274
4,ABT,184109645824,25.938797
